# Forecasts and Reanalysis Exploration

## Reanalysis

Lets first take a look at the reanalysis dataset.  
 - There are 54021 .grib files from 10.01.2018 to 01.09.2024.  
 - There are files for every hour of the day from 00 to 23.  
 - The first complete day is 11.01.2018 and the last complete day is 31.08.2024.  

In [1]:
from pathlib import Path
import xarray as xr
import cfgrib
from IPython.display import display

In [2]:
ICON_REA_PATH = Path("/hpc/rwork2/evalpp/data/rea_grid_0037_R03B07")
def sample_one_file(path, num):
    grib_files = list(path.rglob("*.grib"))
    print(f"found {len(grib_files)} .grib files")
    return grib_files[num]

In [3]:
grib_file_path = sample_one_file(ICON_REA_PATH, 1000)
grib_file_path

found 54021 .grib files


PosixPath('/hpc/rwork2/evalpp/data/rea_grid_0037_R03B07/2023/12/11/rea_grid_0037_R03B07_2023121122.grib')

### Variable Mapping
In the weatherbench2 dataset we consider these forecasted variables which we can map to the new variables using
  - 2m_temperature -> t2m
  - 10m_wind_speed -> fg10

In [4]:
xr.set_options(use_new_combine_kwarg_defaults=True)
dss = cfgrib.open_datasets(grib_file_path, backend_kwargs={'indexpath': ''})

ecCodes provides no latitudes/longitudes for gridType='unstructured_grid'


In [5]:
for ds in dss:
    print(list(ds.coords))
    long_names = {var: ds[var].attrs.get('long_name', None) for var in ds.data_vars}
    print(long_names)
    print("="*60)

['time', 'step', 'depthBelowLandLayer', 'valid_time']
{'W_SO': 'Column-integrated Soil Moisture (multilayers)'}
['time', 'step', 'heightAboveGround', 'valid_time']
{'u10': '10 metre U wind component', 'v10': '10 metre V wind component', 'fg10': 'Time-maximum 10 metre wind gust'}
['time', 'step', 'heightAboveGround', 'valid_time']
{'t2m': '2 metre temperature', 'd2m': '2 metre dewpoint temperature', 'mx2t': 'Time-maximum 2 metre temperature', 'mn2t': 'Time-minimum 2 metre temperature'}
['time', 'step', 'isobaricInhPa', 'valid_time']
{'z': 'Geopotential', 't': 'Temperature', 'u': 'U component of wind', 'v': 'V component of wind', 'r': 'Relative humidity'}
['time', 'step', 'isobaricInhPa', 'valid_time']
{'w': 'Vertical velocity'}
['time', 'step', 'isobaricLayer', 'valid_time']
{'CLCH': 'Cloud Cover (0 - 400 hPa)'}
['time', 'step', 'isobaricLayer', 'valid_time']
{'CLCM': 'Cloud Cover (400 - 800 hPa)'}
['time', 'step', 'meanSea', 'valid_time']
{'prmsl': 'Pressure reduced to MSL'}
['time', '

### Time Steps

Lets explore the time steps and figure out what the comment "ignore timestep 0" means.  
There are steps from 0h to 3h where the first (0h) and last step (3h) coincide.  
We will use the 3h here and ignore timestep 0h as it is not even in the data.

In [50]:
def get_time_info(grib_file_path: Path):
    dss = cfgrib.open_datasets(grib_file_path, backend_kwargs={'indexpath': ''})
    ds = dss[0] # they should all contain the same time info -> only look at the first variable
    time = ds.time.values
    step = ds.step.values.astype('timedelta64[h]')
    valid_time = ds.valid_time.values
    print(f"{grib_file_path=}")
    print(f"{time=}")
    print(f"{step=}")
    print(f"{valid_time=}")
    assert time+step == valid_time

In [49]:
sample_day = Path("/hpc/rwork2/evalpp/data/rea_grid_0037_R03B07") / "2024" / "01" / "01"
grib_files_path = sorted(list(sample_day.glob("*.grib")))

In [51]:
for f in grib_files_path:
    get_time_info(f)
    print("=" * 60)

grib_file_path=PosixPath('/hpc/rwork2/evalpp/data/rea_grid_0037_R03B07/2024/01/01/rea_grid_0037_R03B07_2024010100.grib')
time=np.datetime64('2023-12-31T21:00:00.000000000')
step=np.timedelta64(3,'h')
valid_time=np.datetime64('2024-01-01T00:00:00.000000000')
grib_file_path=PosixPath('/hpc/rwork2/evalpp/data/rea_grid_0037_R03B07/2024/01/01/rea_grid_0037_R03B07_2024010101.grib')
time=np.datetime64('2024-01-01T00:00:00.000000000')
step=np.timedelta64(1,'h')
valid_time=np.datetime64('2024-01-01T01:00:00.000000000')
grib_file_path=PosixPath('/hpc/rwork2/evalpp/data/rea_grid_0037_R03B07/2024/01/01/rea_grid_0037_R03B07_2024010102.grib')
time=np.datetime64('2024-01-01T00:00:00.000000000')
step=np.timedelta64(2,'h')
valid_time=np.datetime64('2024-01-01T02:00:00.000000000')
grib_file_path=PosixPath('/hpc/rwork2/evalpp/data/rea_grid_0037_R03B07/2024/01/01/rea_grid_0037_R03B07_2024010103.grib')
time=np.datetime64('2024-01-01T00:00:00.000000000')
step=np.timedelta64(3,'h')
valid_time=np.datetime64('

## Lets take a look at the ICON EU Ensemble Forecast

The ensemble contains 40 members and there is a forecast for every 3h initialized each day at 00:00 and 12:00.  
We are only interested in the forecasts for lead times of 1,2,3,4, and 5 days (i.e.: 24h, 48h, 72h, 96h, 120h).  
Also for the forecats we only need the mean and standard deviation. Which would lead to a total dataset size of ~500GB.  

In [23]:
ICON_EU_ENS_PATH = Path("/hpc/rwork2/evalpp/data/ICON_EU_EPS")

In [26]:
eu_ens_file = sample_one_file(ICON_EU_ENS_PATH, 1)

found 10439589 .grib files


In [52]:
dss = cfgrib.open_datasets(eu_ens_file, backend_kwargs={'indexpath': ''})
for ds in dss:
    display(ds)

<xarray.Dataset> Size: 5MB
Dimensions:              (depthBelowLandLayer: 8, values: 164984)
Coordinates:
  * depthBelowLandLayer  (depthBelowLandLayer) float64 64B 0.0 0.01 ... 7.29
    number               int64 8B 25
    time                 datetime64[ns] 8B 2023-10-13
    step                 timedelta64[ns] 8B 2 days 12:00:00
    valid_time           datetime64[ns] 8B ...
Dimensions without coordinates: values
Data variables:
    W_SO                 (depthBelowLandLayer, values) float32 5MB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             edzw
    GRIB_centreDescription:  Offenbach
    GRIB_subCentre:          255
    Conventions:             CF-1.7
    institution:             Offenbach

<xarray.Dataset> Size: 2MB
Dimensions:            (values: 164984)
Coordinates:
    number             int64 8B 25
    time               datetime64[ns] 8B 2023-10-13
    step               timedelta64[ns] 8B 2 days 12:00:00
    heightAboveGround  float64 8B 10.0
    valid_time         datetime64[ns] 8B ...
Dimensions without coordinates: values
Data variables:
    u10                (values) float32 660kB ...
    v10                (values) float32 660kB ...
    fg10               (values) float32 660kB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             edzw
    GRIB_centreDescription:  Offenbach
    GRIB_subCentre:          255
    Conventions:             CF-1.7
    institution:             Offenbach

<xarray.Dataset> Size: 3MB
Dimensions:            (values: 164984)
Coordinates:
    number             int64 8B 25
    time               datetime64[ns] 8B 2023-10-13
    step               timedelta64[ns] 8B 2 days 12:00:00
    heightAboveGround  float64 8B 2.0
    valid_time         datetime64[ns] 8B ...
Dimensions without coordinates: values
Data variables:
    t2m                (values) float32 660kB ...
    d2m                (values) float32 660kB ...
    mx2t               (values) float32 660kB ...
    mn2t               (values) float32 660kB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             edzw
    GRIB_centreDescription:  Offenbach
    GRIB_subCentre:          255
    Conventions:             CF-1.7
    institution:             Offenbach

<xarray.Dataset> Size: 26MB
Dimensions:        (isobaricInhPa: 8, values: 164984)
Coordinates:
  * isobaricInhPa  (isobaricInhPa) float64 64B 1e+03 950.0 925.0 ... 400.0 300.0
    number         int64 8B 25
    time           datetime64[ns] 8B 2023-10-13
    step           timedelta64[ns] 8B 2 days 12:00:00
    valid_time     datetime64[ns] 8B ...
Dimensions without coordinates: values
Data variables:
    z              (isobaricInhPa, values) float32 5MB ...
    t              (isobaricInhPa, values) float32 5MB ...
    u              (isobaricInhPa, values) float32 5MB ...
    v              (isobaricInhPa, values) float32 5MB ...
    r              (isobaricInhPa, values) float32 5MB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             edzw
    GRIB_centreDescription:  Offenbach
    GRIB_subCentre:          255
    Conventions:             CF-1.7
    institution:             Offenbach

<xarray.Dataset> Size: 660kB
Dimensions:        (values: 164984)
Coordinates:
    number         int64 8B 25
    time           datetime64[ns] 8B 2023-10-13
    step           timedelta64[ns] 8B 2 days 12:00:00
    isobaricInhPa  float64 8B 500.0
    valid_time     datetime64[ns] 8B ...
Dimensions without coordinates: values
Data variables:
    w              (values) float32 660kB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             edzw
    GRIB_centreDescription:  Offenbach
    GRIB_subCentre:          255
    Conventions:             CF-1.7
    institution:             Offenbach

...

<xarray.Dataset> Size: 660kB
Dimensions:     (values: 164984)
Coordinates:
    number      int64 8B 25
    time        datetime64[ns] 8B 2023-10-13
    step        timedelta64[ns] 8B 2 days 12:00:00
    meanSea     float64 8B 0.0
    valid_time  datetime64[ns] 8B ...
Dimensions without coordinates: values
Data variables:
    prmsl       (values) float32 660kB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             edzw
    GRIB_centreDescription:  Offenbach
    GRIB_subCentre:          255
    Conventions:             CF-1.7
    institution:             Offenbach

<xarray.Dataset> Size: 1MB
Dimensions:     (values: 164984)
Coordinates:
    number      int64 8B 25
    time        datetime64[ns] 8B 2023-10-13
    step        timedelta64[ns] 8B 2 days 12:00:00
    nominalTop  float64 8B 0.0
    valid_time  datetime64[ns] 8B ...
Dimensions without coordinates: values
Data variables:
    avg_tnswrf  (values) float32 660kB ...
    avg_tnlwrf  (values) float32 660kB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             edzw
    GRIB_centreDescription:  Offenbach
    GRIB_subCentre:          255
    Conventions:             CF-1.7
    institution:             Offenbach

<xarray.Dataset> Size: 13MB
Dimensions:      (values: 164984)
Coordinates:
    number       int64 8B 25
    time         datetime64[ns] 8B 2023-10-13
    step         timedelta64[ns] 8B 2 days 12:00:00
    surface      float64 8B 0.0
    valid_time   datetime64[ns] 8B ...
Dimensions without coordinates: values
Data variables: (12/19)
    fsr          (values) float32 660kB ...
    sde          (values) float32 660kB ...
    sd           (values) float32 660kB ...
    crr          (values) float32 660kB ...
    lsrr         (values) float32 660kB ...
    tp           (values) float32 660kB ...
    ...           ...
    T_G          (values) float32 660kB ...
    TQV          (values) float32 660kB ...
    TQI          (values) float32 660kB ...
    CLCT         (values) float32 660kB ...
    TQC          (values) float32 660kB ...
    LPI_CON_MAX  (values) float32 660kB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             edzw
    GRIB_centreDescription:  Offenbach
    GRIB_subCentre:          255
    Conventions:             CF-1.7
    institution:             Offenbach

<xarray.Dataset> Size: 2MB
Dimensions:     (values: 164984)
Coordinates:
    number      int64 8B 25
    time        datetime64[ns] 8B 2023-10-13
    step        timedelta64[ns] 8B 2 days 12:00:00
    level       float64 8B 0.0
    valid_time  datetime64[ns] 8B ...
Dimensions without coordinates: values
Data variables:
    HBAS_CON    (values) float32 660kB ...
    HTOP_CON    (values) float32 660kB ...
    CAPE_ML     (values) float32 660kB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             edzw
    GRIB_centreDescription:  Offenbach
    GRIB_subCentre:          255
    Conventions:             CF-1.7
    institution:             Offenbach

<xarray.Dataset> Size: 660kB
Dimensions:     (values: 164984)
Coordinates:
    number      int64 8B 25
    time        datetime64[ns] 8B 2023-10-13
    step        timedelta64[ns] 8B 2 days 12:00:00
    level       float64 8B 800.0
    valid_time  datetime64[ns] 8B ...
Dimensions without coordinates: values
Data variables:
    CLCL        (values) float32 660kB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             edzw
    GRIB_centreDescription:  Offenbach
    GRIB_subCentre:          255
    Conventions:             CF-1.7
    institution:             Offenbach

### Variable mapping

In [53]:
for ds in dss:
    print(list(ds.coords))
    long_names = {var: ds[var].attrs.get('long_name', None) for var in ds.data_vars}
    print(long_names)
    print("="*60)

['number', 'time', 'step', 'depthBelowLandLayer', 'valid_time']
{'W_SO': 'Column-integrated Soil Moisture (multilayers)'}
['number', 'time', 'step', 'heightAboveGround', 'valid_time']
{'u10': '10 metre U wind component', 'v10': '10 metre V wind component', 'fg10': 'Time-maximum 10 metre wind gust'}
['number', 'time', 'step', 'heightAboveGround', 'valid_time']
{'t2m': '2 metre temperature', 'd2m': '2 metre dewpoint temperature', 'mx2t': 'Time-maximum 2 metre temperature', 'mn2t': 'Time-minimum 2 metre temperature'}
['number', 'time', 'step', 'isobaricInhPa', 'valid_time']
{'z': 'Geopotential', 't': 'Temperature', 'u': 'U component of wind', 'v': 'V component of wind', 'r': 'Relative humidity'}
['number', 'time', 'step', 'isobaricInhPa', 'valid_time']
{'w': 'Vertical velocity'}
['number', 'time', 'step', 'isobaricLayer', 'valid_time']
{'CLCH': 'Cloud Cover (0 - 400 hPa)'}
['number', 'time', 'step', 'isobaricLayer', 'valid_time']
{'CLCM': 'Cloud Cover (400 - 800 hPa)'}
['number', 'time', 

# Grid Files
For completeness lets check out the grid file here as well.
From grib_dump we know that numberOfGridUsed = 37.

In [19]:
gridfile_path = "/hpc/rhome/routfox/routfox/icon/grids/public/edzw/icon_grid_0037_R03B07_N02.nc"
ds_grid = xr.open_dataset(gridfile_path)

## Also lets take a look at the created remap file

In [21]:
remapfile_path = "/hpc/uhome/extmfeik/GenPP/src/genpp/data/icon/remap_grid/remap_gennn.nc"
ds_remap = xr.open_dataset(remapfile_path)

In [22]:
ds_remap

<xarray.Dataset> Size: 7MB
Dimensions:              (src_grid_rank: 1, dst_grid_rank: 2,
                          src_grid_size: 164984, dst_grid_size: 61680,
                          num_links: 61680, num_wgts: 1)
Dimensions without coordinates: src_grid_rank, dst_grid_rank, src_grid_size,
                                dst_grid_size, num_links, num_wgts
Data variables: (12/13)
    src_grid_dims        (src_grid_rank) int32 4B ...
    dst_grid_dims        (dst_grid_rank) int32 8B ...
    src_grid_center_lat  (src_grid_size) float64 1MB ...
    dst_grid_center_lat  (dst_grid_size) float64 493kB ...
    src_grid_center_lon  (src_grid_size) float64 1MB ...
    dst_grid_center_lon  (dst_grid_size) float64 493kB ...
    ...                   ...
    dst_grid_imask       (dst_grid_size) int32 247kB ...
    src_grid_frac        (src_grid_size) float64 1MB ...
    dst_grid_frac        (dst_grid_size) float64 493kB ...
    src_address          (num_links) int32 247kB ...
    dst_address          (num_links) int32 247kB ...
    remap_matrix         (num_links, num_wgts) float64 493kB ...
Attributes:
    title:          CDO remapping
    normalization:  none
    map_method:     Nearest neighbor
    conventions:    SCRIP
    source_grid:    unstructured
    dest_grid:      projection
    history:        01 Jan 2026 : cdo gennn,target_grid.txt /hpc/rhome/routfo...
    CDO:            Climate Data Operators version 2.5.2 (https://mpimet.mpg....

## Test

Open the remapped reanalysis file

In [7]:
ds_rot = xr.open_dataset("../cdo_scripts/rea_2020_April_00_12.nc")

In [8]:
ds_rot

<xarray.Dataset> Size: 1GB
Dimensions:       (plev: 1, bnds: 2, plev_2: 1, plev_3: 1, depth: 8, time: 60,
                   y: 240, x: 260, lev: 1, plev_4: 8, plev_5: 1, height: 1,
                   height_2: 1)
Coordinates:
  * plev          (plev) float64 8B 0.0
  * plev_2        (plev_2) float64 8B 8e+04
  * plev_3        (plev_3) float64 8B 4e+04
  * depth         (depth) float64 64B 0.0 0.01 0.03 0.09 0.27 0.81 2.43 7.29
  * time          (time) datetime64[ns] 480B 2020-04-01 ... 2020-04-30T12:00:00
  * y             (y) float64 2kB -15.0 -14.87 -14.74 ... 15.81 15.94 16.07
  * x             (x) float64 2kB -16.64 -16.51 -16.38 ... 16.77 16.9 17.03
  * lev           (lev) float64 8B 0.0
  * plev_4        (plev_4) float64 64B 3e+04 4e+04 5e+04 ... 9.5e+04 1e+05
  * plev_5        (plev_5) float64 8B 5e+04
  * height        (height) float64 8B 2.0
  * height_2      (height_2) float64 8B 10.0
Dimensions without coordinates: bnds
Data variables: (12/47)
    rotated_pole  int32 4B ...
    plev_bnds     (plev, bnds) float64 16B ...
    plev_2_bnds   (plev_2, bnds) float64 16B ...
    plev_3_bnds   (plev_3, bnds) float64 16B ...
    depth_bnds    (depth, bnds) float64 128B ...
    ALB_RAD       (time, y, x) float32 15MB ...
    ...            ...
    V             (time, plev_4, y, x) float32 120MB ...
    VMAX_10M      (time, height_2, y, x) float32 15MB ...
    V_10M         (time, height_2, y, x) float32 15MB ...
    W_SNOW        (time, y, x) float32 15MB ...
    W_SO          (time, depth, y, x) float32 120MB ...
    Z0            (time, y, x) float32 15MB ...
Attributes:
    CDI:          Climate Data Interface version 2.5.2 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    institution:  Deutscher Wetterdienst
    history:      Thu Jan 08 23:36:36 2026: cdo -f nc4 -z zip_6 remap,/hpc/uh...
    CDO:          Climate Data Operators version 2.5.2 (https://mpimet.mpg.de...